In [ ]:
from google.colab import drive
import shutil
import os

# Mount Google Drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

### Unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "principled-intelligence/Qwen3.5-4B-text-only",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


ImportError: Unsloth: Your transformers version of 4.56.2 does not support Qwen3.5.
The minimum required version is 5.2.0.
Try `pip install --upgrade "transformers>=5.2.0"`
to obtain the latest transformers build, then restart this session.

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 32,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.7.1 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the `Qwen-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Qwen-3 renders multi turn conversations like below:

```
<|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>

```
We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen3-instruct",
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("AlicanKiraz0/Cybersecurity-Dataset-Fenrir-v2.1", split = "train")

README.md: 0.00B [00:00, ?B/s]

220426-CyberSec-Dataset_escaped.jsonl:   0%|          | 0.00/431M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

We now use `standardize_data_formats` to try converting datasets to the correct format for finetuning purposes!

In [ ]:
from unsloth.chat_templates import standardize_data_formats
dataset = standardize_data_formats(dataset)

Let's see how row 100 looks like!

In [ ]:
dataset[100]

{'system': 'You are an advanced AI assistant specialized in cybersecurity causal reasoning and threat analysis. Your expertise encompasses offensive security, defensive strategies, incident response, threat intelligence, and systemic security analysis across all technology domains. CORE CYBERSECURITY CAPABILITIES: - Deep understanding of attack chains, kill chains, and threat actor behaviors - Analysis of vulnerability-to-exploit causal relationships - Recognition of security control effectiveness and bypass mechanisms - Incident cascade analysis and lateral movement patterns - Risk quantification and threat modeling expertise - Understanding of human factors in security failures RESPONSE STRUCTURE: For each cybersecurity causal reasoning question, provide a comprehensive analysis following this exact format: ## Security Causal Analysis **Direct Answer:** [Concise 1-2 sentence conclusion addressing the core security question] ### Primary Attack/Defense Mechanisms: [Explain the main cau

We now have to apply the chat template for `Qwen-3` onto the conversations, and save it to `text`.

In [ ]:
def formatting_prompts_func(examples):
    conversations = [
        [
            {"role": "system", "content": s},
            {"role": "user", "content": u},
            {"role": "assistant", "content": a},
        ]
        for s, u, a in zip(
            examples["system"],
            examples["user"],
            examples["assistant"],
        )
    ]

    return {
        "text": tokenizer.apply_chat_template(
            conversations,
            tokenize=False,
            add_generation_prompt=False,
        )
    }

dataset = dataset.map(formatting_prompts_func, batched=True)

Map:   0%|          | 0/99870 [00:00<?, ? examples/s]

Let's see how the chat template did!

In [ ]:
dataset[100]['text']

"<|im_start|>system\nYou are an advanced AI assistant specialized in cybersecurity causal reasoning and threat analysis. Your expertise encompasses offensive security, defensive strategies, incident response, threat intelligence, and systemic security analysis across all technology domains. CORE CYBERSECURITY CAPABILITIES: - Deep understanding of attack chains, kill chains, and threat actor behaviors - Analysis of vulnerability-to-exploit causal relationships - Recognition of security control effectiveness and bypass mechanisms - Incident cascade analysis and lateral movement patterns - Risk quantification and threat modeling expertise - Understanding of human factors in security failures RESPONSE STRUCTURE: For each cybersecurity causal reasoning question, provide a comprehensive analysis following this exact format: ## Security Causal Analysis **Direct Answer:** [Concise 1-2 sentence conclusion addressing the core security question] ### Primary Attack/Defense Mechanisms: [Explain the

<a name="Train"></a>
### Train the model
Now let's train our model. We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    eval_dataset = None, # Can set up evaluation!
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4, # Use GA to mimic batch size!
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4, # Reduce to 2e-5 for long training runs
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/99870 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 99,870 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,2.671700
2,2.943900
3,2.778100
4,2.244500
5,2.279100
6,2.088100
7,1.749200
8,1.812800
9,1.819300
10,1.805400


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Qwen-3` team, the recommended settings for instruct inference are `temperature = 0.7, top_p = 0.8, top_k = 20`

For reasoning chat based inference, `temperature = 0.6, top_p = 0.95, top_k = 20`

In [ ]:
messages = [

            {"role": "system", "content":" You are an advanced AI assistant specialized in cybersecurity causal reasoning and threat analysis. Your expertise encompasses offensive security, defensive strategies, incident response, threat intelligence, and systemic security analysis across all technology domains. CORE CYBERSECURITY CAPABILITIES: - Deep understanding of attack chains, kill chains, and threat actor behaviors - Analysis of vulnerability-to-exploit causal relationships - Recognition of security control effectiveness and bypass mechanisms - Incident cascade analysis and lateral movement patterns - Risk quantification and threat modeling expertise - Understanding of human factors in security failures RESPONSE STRUCTURE: For each cybersecurity causal reasoning question, provide a comprehensive analysis following this exact format: ## Security Causal Analysis **Direct Answer:** [Concise 1-2 sentence conclusion addressing the core security question] ### Primary Attack/Defense Mechanisms: [Explain the main causal pathways in the security context] 1. [Initial vector/vulnerability → exploitation mechanism] 2. [Propagation/escalation pathway if applicable] 3. [Impact chain and cascading effects] [Include technical details and TTPs (Tactics, Techniques, Procedures)] ### Evidence & Threat Intelligence: - **Confirmed/Documented:** [CVEs, security research, incident reports, vendor advisories] - **Observed in Wild:** [Threat intel, honeypot data, OSINT findings] - **Theoretical/PoC:** [Security research, responsible disclosure, lab demonstrations] ### Temporal Attack Dynamics: - **Initial Compromise:** [0-24 hours: reconnaissance, initial access] - **Establishment Phase:** [1-30 days: persistence, privilege escalation] - **Operations Phase:** [30+ days: lateral movement, data exfiltration] - **Detection Windows:** [Mean time to detect, dwell time statistics] ### Alternative Attack Vectors: - [Other exploitation methods that could achieve similar outcomes] - [Supply chain or third-party risk considerations] - [Social engineering or insider threat alternatives] ### Security System Interactions: - **Kill Chain Disruption Points:** [Where defensive controls can break the attack] - **Defense Evasion:** [How attackers bypass controls] - **Detection Opportunities:** [Behavioral indicators, anomalies] - **Cascading Failures:** [How one compromise leads to others] ### Risk Quantification: - **CVSS/EPSS Scores:** [If applicable] - **Likelihood Assessment:** [Based on threat landscape] - **Impact Analysis:** [CIA triad, business impact] - **Attack Complexity:** [Required skill level, resources] ### Uncertainties & Intelligence Gaps: - [Unknown vulnerabilities (0-days)] - [Attribution challenges] - [Evolving TTPs] - [Environmental dependencies] ### Security Recommendations: - **Preventive Controls:** [Hardening, patching, configuration] - **Detective Controls:** [Monitoring, SIEM rules, threat hunting] - **Response Strategies:** [Incident response, containment, recovery] - **Compensating Controls:** [When primary controls fail] **Threat Assessment Level:** [Critical/High/Medium/Low] with justification CYBERSECURITY-SPECIFIC GUIDELINES: 1. Apply the principle of least privilege and zero trust concepts 2. Consider the full MITRE ATT&CK framework for comprehensive analysis 3. Account for both technical and human factor vulnerabilities 4. Analyze defense-in-depth strategies and their effectiveness 5. Include supply chain and third-party risks in the analysis 6. Consider both nation-state and criminal threat actors 7. Address compliance and regulatory implications where relevant 8. Evaluate emerging threats (AI-powered attacks, quantum computing risks) 9. Include cloud-specific and hybrid environment considerations 10. Account for IoT/OT security implications in relevant scenarios DOMAIN-SPECIFIC SECURITY CONSIDERATIONS: - **Network Security:** OSI layer interactions, protocol vulnerabilities, segmentation - **Application Security:** OWASP Top 10, secure SDLC, code vulnerabilities - **Cloud Security:** Shared responsibility, misconfigurations, multi-tenancy risks - **Identity & Access:** Authentication chains, privilege escalation, federation risks - **Cryptography:** Algorithm weaknesses, implementation flaws, key management - **Physical Security:** Environmental threats, hardware tampering, side-channels - **Operational Security:** Process failures, insider threats, social engineering THREAT ACTOR CONSIDERATIONS: - **APT Groups:** Nation-state capabilities, persistence, resources - **Cybercriminals:** Ransomware operations, financial motivation - **Hacktivists:** Ideological targeting, public impact focus - **Insider Threats:** Privileged access abuse, data theft - **Supply Chain:** Third-party compromises, software dependencies Remember: In cybersecurity, assume breach and analyze both prevention and detection/response. Consider that attackers need only one success while defenders must succeed consistently."},
            {"role": "user", "content": "In which scenarios might attackers leverage edge cases of Building automated response playbooks for ransomware incidents to bypass existing controls, and how can purple‐team exercises uncover such blind spots?"},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer(text, return_tensors = "pt").to("cuda"),
    max_new_tokens = 1000, # Increase for longer outputs!
    temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

## Security Causal Analysis **Direct Answer:** Attackers can bypass automated response playbooks by exploiting edge cases in their logic, such as unhandled error conditions, incomplete coverage of attack vectors, or lack of integration with external systems. Purple-team exercises uncover these blind spots by simulating realistic attack scenarios that test playbook effectiveness under stress, revealing gaps in detection, response, and recovery mechanisms. ### Primary Attack/Defense Mechanisms: 1. **Edge Case Exploitation:** Attackers target inputs or conditions that are not explicitly covered in automated playbooks, such as unusual network traffic patterns, encrypted file naming conventions, or specific error messages that trigger false positives. 2. **Logic Gaps:** Automated playbooks may lack conditional branches for complex scenarios, leading to missed detections or inappropriate responses. 3. **Integration Failures:** Playbooks may not integrate with external security tools or syste

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Hugging Face's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("qwen_lora_Cybersecucity")  # Local saving
tokenizer.save_pretrained("qwen_lora_Cybersecucity")
# model.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving
# tokenizer.push_to_hub("your_name/qwen_lora", token = "YOUR_HF_TOKEN") # Online saving

('qwen_lora_Cybersecucity/tokenizer_config.json',
 'qwen_lora_Cybersecucity/special_tokens_map.json',
 'qwen_lora_Cybersecucity/chat_template.jinja',
 'qwen_lora_Cybersecucity/vocab.json',
 'qwen_lora_Cybersecucity/merges.txt',
 'qwen_lora_Cybersecucity/added_tokens.json',
 'qwen_lora_Cybersecucity/tokenizer.json')

In [ ]:

# Source folder (change if your model is saved elsewhere)
source_dir = "/content/qwen_lora_Cybersecucity"

# Destination folder in Drive
dest_dir = "/content/drive/MyDrive/qwen_Cybersecucity"

# Remove old copy if it exists
if os.path.exists(dest_dir):
    shutil.rmtree(dest_dir)

# Copy model folder to Drive
shutil.copytree(source_dir, dest_dir)

print(f"Copied successfully to: {dest_dir}")

Copied successfully to: /content/drive/MyDrive/qwen_Cybersecucity


In [ ]:
model.push_to_hub_gguf("basilmh25/qwen_Cybersecucity", tokenizer, quantization_method = "q4_k_m", token = "**************************************")


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...instruct-2507.Q4_K_M.gguf:   0%|          |  548kB / 2.50GB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/basilmh25/qwen_Cybersecucity
Unsloth: Cleaning up temporary files...


'basilmh25/qwen_Cybersecucity'